# Phase 2 — Preprocessing multi-sources réaliste

Objectif : construire un dataset d'entraînement plus diversifié pour la détection d'intents e-commerce.

Sources utilisées :
- `bitext-retail.csv` : référence principale.
- `Bitext_Customer_Support.csv` : enrichissement support client.
- `synthetic_ecommerce_data.csv` : intents e-commerce supplémentaires si compatibles.
- `Ecommerce_FAQ_intents.json` : FAQ/chatbot intents si format compatible.
- Amazon QA : `multi_questions.csv`, `single_qna.csv` utilisés uniquement via mapping heuristique haute confiance.

Sources éliminées ou non utilisées directement :
- `multi_answers.csv` : réponses Amazon, non utilisées pour l'intent classification.
- toutes les lignes dont l'intent ne peut pas être mappé vers l'espace d'intents du projet.


## 1. Imports & configuration

In [14]:
import pandas as pd
import numpy as np
import re
import json
import random
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PROJECT_DIR = Path("D:/conv_nlp_pipeline")

DATA_RAW = PROJECT_DIR / "data" / "raw"
DATA_PROC = PROJECT_DIR / "data" / "processed"
DATA_SPLITS = PROJECT_DIR / "data" / "splits"

DATA_PROC.mkdir(parents=True, exist_ok=True)
DATA_SPLITS.mkdir(parents=True, exist_ok=True)

print("DATA_RAW   :", DATA_RAW)
print("DATA_PROC  :", DATA_PROC)
print("DATA_SPLITS:", DATA_SPLITS)


DATA_RAW   : D:\conv_nlp_pipeline\data\raw
DATA_PROC  : D:\conv_nlp_pipeline\data\processed
DATA_SPLITS: D:\conv_nlp_pipeline\data\splits


## 2. Fonctions utilitaires : colonnes, normalisation, mapping

In [15]:
TEXT_COL_CANDIDATES = [
    "instruction", "text", "query", "question", "utterance",
    "input", "user_message", "customer_query", "prompt","Question",
    "QuestionText"
]
RESP_COL_CANDIDATES = [
    "response", "answer", "completion", "assistant_response", "agent_response",    "Answer"

]
INTENT_COL_CANDIDATES = [
    "intent", "label", "category_label", "tag", "intent_name"
]
CATEGORY_COL_CANDIDATES = ["category", "domain", "group","Category"]
FLAGS_COL_CANDIDATES = ["flags", "tags"]

INTENT_ALIASES = {
    "get_refund": "request_refund",
    "refund_request": "request_refund",
    "refund": "request_refund",
    "return": "return_product",
    "return_item": "return_product",
    "track_shipping": "track_delivery",
    "shipping_status": "track_delivery",
    "delivery_status": "track_delivery",
    "track_package": "track_delivery",
    "where_is_my_order": "track_order",
    "product_info": "product_information",
    "item_information": "product_information",
    "product_details": "product_information",
    "product_query": "product_information",
    "speak_to_agent": "human_agent",
    "contact_agent": "human_agent",
    "talk_to_human": "human_agent",
    "contact_support": "customer_service",
    "invoice_request": "request_invoice",
    "get_invoice": "request_invoice",
    "payment_method": "payment_methods",
}

HEURISTIC_INTENT_RULES = [
    (r"\b(refund policy|refund rules)\b", "refund_policy"),
    (r"\b(refund status|where.*refund|track.*refund)\b", "refund_status"),
    (r"\b(refund|money back|reimburse)\b", "request_refund"),

    (r"\b(return policy|return rules)\b", "return_policy"),
    (r"\b(return in store|store return)\b", "return_product_in_store"),
    (r"\b(return online|online return)\b", "return_product_online"),
    (r"\b(return|send back|returning)\b", "return_product"),

    (r"\b(shipping cost|shipping fee|delivery fee|delivery cost)\b", "shipping_costs"),
    (r"\b(delivery time|arrive|arrival|how long.*deliver|when.*deliver|shipping time)\b", "delivery_time"),
    (r"\b(track|tracking|where.*order|where.*package|package status|shipment status)\b", "track_delivery"),

    (r"\b(broken|defective|not working|doesn.?t work|faulty)\b", "product_issue"),
    (r"\b(damaged|arrived broken|arrived damaged)\b", "damaged_delivery"),
    (r"\b(missing|not included|didn.?t receive)\b", "missing_item"),
    (r"\b(wrong item|wrong product|different item)\b", "wrong_item"),

    (r"\b(available|availability|in stock|out of stock|stock)\b", "availability"),
    (r"\b(size|dimension|material|weight|color|compatible|fit|specification|details|feature|features)\b", "product_information"),

    (r"\b(payment issue|payment failed|charged twice|card declined|transaction failed)\b", "payment_issue"),
    (r"\b(payment methods|pay methods|accepted cards|paypal|credit card)\b", "payment_methods"),
    (r"\b(pay|checkout|purchase)\b", "pay"),

    (r"\b(invoice|receipt|bill)\b", "request_invoice"),
    (r"\b(password|login|log in|sign in|account access)\b", "recover_password"),
    (r"\b(contact support|customer service|support team|help center)\b", "customer_service"),
    (r"\b(human agent|real person|operator|representative)\b", "human_agent"),
]

def normalize_intent_name(x):
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    x = re.sub(r"[\s\-]+", "_", x)
    x = re.sub(r"[^a-z0-9_]+", "", x)
    return INTENT_ALIASES.get(x, x)


def normalize_text(text):
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s([?.!,;:])", r"\1", text)
    return text


def first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def infer_intent_heuristic(text):
    text = str(text).lower()
    for pattern, intent in HEURISTIC_INTENT_RULES:
        if re.search(pattern, text):
            return intent
    return None


def standardize_dataset(df, source_name, require_intent=True, use_heuristic_if_no_intent=False):
    df = df.copy()
    text_col = first_existing_col(df, TEXT_COL_CANDIDATES)
    resp_col = first_existing_col(df, RESP_COL_CANDIDATES)
    intent_col = first_existing_col(df, INTENT_COL_CANDIDATES)
    cat_col = first_existing_col(df, CATEGORY_COL_CANDIDATES)
    flags_col = first_existing_col(df, FLAGS_COL_CANDIDATES)

    if text_col is None:
        print(f"⚠ {source_name}: aucune colonne texte trouvée → ignoré")
        return pd.DataFrame()

    out = pd.DataFrame()
    out["instruction"] = df[text_col].astype(str)
    out["response"] = df[resp_col].astype(str) if resp_col else ""

    if intent_col is not None:
        out["intent"] = df[intent_col].apply(normalize_intent_name)
    elif use_heuristic_if_no_intent:
        out["intent"] = out["instruction"].apply(infer_intent_heuristic)
    elif require_intent:
        print(f"⚠ {source_name}: aucune colonne intent trouvée → ignoré")
        return pd.DataFrame()
    else:
        out["intent"] = None

    out["category"] = df[cat_col].astype(str) if cat_col else source_name.upper()
    out["flags"] = df[flags_col].astype(str) if flags_col else ""
    out["source"] = source_name
    out["instruction"] = out["instruction"].apply(normalize_text)
    out = out[out["instruction"].str.strip().ne("")]
    out = out.dropna(subset=["intent"])
    return out


## 3. Charger Bitext Retail et Bitext Customer Support

In [16]:
retail_path = DATA_RAW / "bitext-retail.csv"
if not retail_path.exists():
    raise FileNotFoundError(f"Fichier obligatoire manquant: {retail_path}")

support_candidates = [
    DATA_RAW / "Bitext_Customer_Support.csv",
    DATA_RAW / "Bitext_Support_Customer.csv",
    DATA_RAW / "bitext-customer-support.csv",
]
support_path = next((p for p in support_candidates if p.exists()), None)
if support_path is None:
    raise FileNotFoundError("Dataset support introuvable: ajoute Bitext_Customer_Support.csv ou Bitext_Support_Customer.csv")

raw_retail = pd.read_csv(retail_path)
raw_support = pd.read_csv(support_path)

df_retail = standardize_dataset(raw_retail, "retail", require_intent=True)
df_support = standardize_dataset(raw_support, "support", require_intent=True)

canonical_intents = sorted(set(df_retail["intent"].unique()) | set(df_support["intent"].unique()))

print(f"Retail : {len(df_retail):,} lignes · {df_retail['intent'].nunique()} intents")
print(f"Support: {len(df_support):,} lignes · {df_support['intent'].nunique()} intents")
print(f"Canonical intents: {len(canonical_intents)}")


Retail : 44,884 lignes · 46 intents
Support: 26,872 lignes · 27 intents
Canonical intents: 66


## 4. Charger les datasets e-commerce supplémentaires

In [17]:
extra_dfs = []
rejected_stats = []
INTENT_ALIASES.update({
    "order_tracking": "track_order",
    "track_my_order": "track_order",
    "shipping_tracking": "track_delivery",
    "package_tracking": "track_delivery",

    "cancel": "cancel_order",
    "cancel_purchase": "cancel_order",

    "refund_status": "refund_status",
    "refund_policy": "refund_policy",

    "product_availability": "availability",
    "stock_check": "availability",
    "in_stock": "availability",

    "product_question": "product_information",
    "faq_product": "product_information",

    "contact": "customer_service",
    "support": "customer_service",
    "agent": "human_agent",

    "checkout": "pay",
    "make_payment": "pay",
    "payment_problem": "payment_issue",

    "return_policy": "return_policy",
    "return_request": "return_product",
})
def keep_canonical(df, source_name):
    before = len(df)
    df = df[df["intent"].isin(canonical_intents)].copy()
    after = len(df)
    rejected_stats.append({"source": source_name, "before": before, "kept": after, "rejected": before - after})
    print(f"{source_name:<28}: gardé {after:>7,}/{before:<7,} lignes")
    return df

synthetic_path = DATA_RAW / "synthetic_ecommerce_data.csv"
if synthetic_path.exists():
    raw_syn = pd.read_csv(synthetic_path)
    df_syn = standardize_dataset(raw_syn, "synthetic_ecommerce", require_intent=True)
    df_syn = keep_canonical(df_syn, "synthetic_ecommerce")
    if len(df_syn):
        extra_dfs.append(df_syn)
else:
    print("ℹ synthetic_ecommerce_data.csv introuvable → ignoré")

faq_path = DATA_RAW / "Ecommerce_FAQ_intents.json"
if faq_path.exists():
    with open(faq_path, "r", encoding="utf-8") as f:
        faq_obj = json.load(f)

    rows = []
    intents_block = faq_obj.get("intents", faq_obj) if isinstance(faq_obj, dict) else faq_obj

    if isinstance(intents_block, list):
        for item in intents_block:
            if not isinstance(item, dict):
                continue
            intent = normalize_intent_name(item.get("tag") or item.get("intent") or item.get("label"))
            category = item.get("category", "ECOMMERCE_FAQ")
            patterns = item.get("patterns") or item.get("questions") or item.get("utterances") or []
            responses = item.get("responses") or item.get("answers") or [""]
            response = responses[0] if isinstance(responses, list) and len(responses) else ""
            if isinstance(patterns, str):
                patterns = [patterns]
            for p in patterns:
                rows.append({
                    "instruction": normalize_text(str(p)),
                    "response": str(response),
                    "intent": intent,
                    "category": category,
                    "flags": "faq_json",
                    "source": "ecommerce_faq"
                })

    df_faq = pd.DataFrame(rows)
    if len(df_faq):
        df_faq = df_faq.dropna(subset=["intent"])
        df_faq = keep_canonical(df_faq, "ecommerce_faq")
        if len(df_faq):
            extra_dfs.append(df_faq)
    else:
        print("⚠ Ecommerce_FAQ_intents.json lu mais aucun pattern exploitable")
else:
    print("ℹ Ecommerce_FAQ_intents.json introuvable → ignoré")


synthetic_ecommerce         : gardé     288/1,005   lignes
ecommerce_faq               : gardé      52/422     lignes


## 5. Ajouter Amazon QA filtré/mappé

In [18]:
amazon_dfs = []

single_path = DATA_RAW / "single_qna.csv"
if single_path.exists():
    raw_single = pd.read_csv(single_path)
    print("single_qna columns:", raw_single.columns.tolist())

    df_single = standardize_dataset(
        raw_single,
        "amazon_single_qna",
        require_intent=False,
        use_heuristic_if_no_intent=True
    )

    if len(df_single) > 0 and "intent" in df_single.columns:
        df_single = keep_canonical(df_single, "amazon_single_qna")
        if len(df_single):
            amazon_dfs.append(df_single)
    else:
        print("amazon_single_qna: aucune ligne exploitable")
else:
    print("ℹ single_qna.csv introuvable → ignoré")


mq_path = DATA_RAW / "multi_questions.csv"
if mq_path.exists():
    raw_mq = pd.read_csv(mq_path)
    print("multi_questions columns:", raw_mq.columns.tolist())

    df_mq = standardize_dataset(
        raw_mq,
        "amazon_multi_questions",
        require_intent=False,
        use_heuristic_if_no_intent=True
    )

    if len(df_mq) > 0 and "intent" in df_mq.columns:
        df_mq = keep_canonical(df_mq, "amazon_multi_questions")
        if len(df_mq):
            amazon_dfs.append(df_mq)
    else:
        print("amazon_multi_questions: aucune ligne exploitable")
else:
    print("ℹ multi_questions.csv introuvable → ignoré")


ma_path = DATA_RAW / "multi_answers.csv"
if ma_path.exists():
    print("ℹ multi_answers.csv trouvé mais ignoré: réponses Amazon, pas des messages utilisateur pour intent classification")


if amazon_dfs:
    df_amazon = pd.concat(amazon_dfs, ignore_index=True)

    MAX_AMAZON = 8000
    if len(df_amazon) > MAX_AMAZON:
        df_amazon = df_amazon.sample(
            n=MAX_AMAZON,
            random_state=RANDOM_SEED
        ).reset_index(drop=True)

    extra_dfs.append(df_amazon)
    print(f"Amazon QA final gardé: {len(df_amazon):,} lignes")
else:
    print("Amazon QA: aucune ligne exploitable")

single_qna columns: ['QuestionType', 'Asin', 'AnswerTime', 'UnixTime', 'Question', 'AnswerType', 'Answer', 'Category']
amazon_single_qna           : gardé 256,156/256,156 lignes
multi_questions columns: ['QuestionID', 'QuestionType', 'Category', 'AskerID', 'QuestionTime', 'QuestionText']
amazon_multi_questions      : gardé  31,762/31,762  lignes
ℹ multi_answers.csv trouvé mais ignoré: réponses Amazon, pas des messages utilisateur pour intent classification
Amazon QA final gardé: 8,000 lignes


## 6. Fusion finale, nettoyage texte et labels

In [19]:
sources = [df_retail, df_support] + extra_dfs
df_all = pd.concat(sources, ignore_index=True)

df_all["instruction_clean"] = df_all["instruction"].apply(normalize_text)
df_all["response_clean"] = df_all["response"].apply(normalize_text)
df_all = df_all.drop_duplicates(subset=["instruction_clean", "intent", "source"]).reset_index(drop=True)
df_all = df_all[df_all["intent"].isin(canonical_intents)].copy()

INTENT_LIST = sorted(df_all["intent"].unique())
LABEL_MAP = {intent: i for i, intent in enumerate(INTENT_LIST)}
ID2LABEL = {i: intent for intent, i in LABEL_MAP.items()}
df_all["label"] = df_all["intent"].map(LABEL_MAP).astype(int)

df_all["word_count"] = df_all["instruction_clean"].str.split().str.len()
df_all["char_count"] = df_all["instruction_clean"].str.len()

external_sources = {"synthetic_ecommerce", "ecommerce_faq", "amazon_single_qna", "amazon_multi_questions"}
df_all["is_hard"] = df_all["source"].isin(external_sources)

print("Résumé sources finales:")
print(df_all["source"].value_counts().to_string())
print()
print(f"Total final : {len(df_all):,} lignes · {df_all['intent'].nunique()} intents")
print(f"Hard samples: {df_all['is_hard'].sum():,}")

with open(DATA_PROC / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(LABEL_MAP, f, indent=2, ensure_ascii=False)
with open(DATA_PROC / "intent_list.json", "w", encoding="utf-8") as f:
    json.dump(INTENT_LIST, f, indent=2, ensure_ascii=False)

pd.DataFrame(rejected_stats).to_csv(DATA_PROC / "source_filtering_stats.csv", index=False)
df_all.to_csv(DATA_PROC / "dataset_all_sources_clean.csv", index=False)
df_all.to_parquet(DATA_PROC / "dataset_all_sources_clean.parquet", index=False)


Résumé sources finales:
source
retail                    44479
support                   24533
amazon_single_qna          6918
amazon_multi_questions      909
synthetic_ecommerce         288
ecommerce_faq                52

Total final : 77,179 lignes · 66 intents
Hard samples: 8,167


## 7. Split : train multi-sources, val Retail + realistic_val, test Retail

In [20]:
COLS = [
    "instruction_clean", "response_clean", "intent", "category",
    "label", "flags", "source", "word_count", "char_count", "is_hard"
]
COLS = [c for c in COLS if c in df_all.columns]

df_retail_clean = df_all[df_all["source"] == "retail"][COLS].copy()

X_trainval, X_test = train_test_split(
    df_retail_clean,
    test_size=0.15,
    stratify=df_retail_clean["label"],
    random_state=RANDOM_SEED
)

X_train_retail, X_val_retail = train_test_split(
    X_trainval,
    test_size=0.176,
    stratify=X_trainval["label"],
    random_state=RANDOM_SEED
)

extra_train = df_all[df_all["source"] != "retail"][COLS].copy()
X_train_base = pd.concat([X_train_retail, extra_train], ignore_index=True)

realistic_val_path = DATA_RAW / "realistic_val.csv"
if realistic_val_path.exists():
    raw_real_val = pd.read_csv(realistic_val_path)
    df_real_val = standardize_dataset(raw_real_val, "realistic_val", require_intent=True)
    df_real_val = df_real_val[df_real_val["intent"].isin(LABEL_MAP.keys())].copy()
    df_real_val["instruction_clean"] = df_real_val["instruction"].apply(normalize_text)
    df_real_val["response_clean"] = df_real_val["response"].apply(normalize_text)
    df_real_val["label"] = df_real_val["intent"].map(LABEL_MAP).astype(int)
    df_real_val["word_count"] = df_real_val["instruction_clean"].str.split().str.len()
    df_real_val["char_count"] = df_real_val["instruction_clean"].str.len()
    df_real_val["is_hard"] = True
    df_real_val = df_real_val[COLS]
    X_val = pd.concat([X_val_retail, df_real_val], ignore_index=True)
    print(f"realistic_val ajouté: {len(df_real_val):,} lignes")
else:
    X_val = X_val_retail.copy()
    print("ℹ realistic_val.csv non trouvé → val = Retail uniquement")

print("Split base créé:")
print(f"  Train base: {len(X_train_base):,}")
print(f"  Val       : {len(X_val):,}")
print(f"  Test      : {len(X_test):,} (Retail uniquement)")


ℹ realistic_val.csv non trouvé → val = Retail uniquement
Split base créé:
  Train base: 63,852
  Val       : 6,655
  Test      : 6,672 (Retail uniquement)


## 8. Augmentation train-only

In [21]:
ABBREVS = {
    "you": "u", "your": "ur", "please": "pls", "because": "bc",
    "are": "r", "thank you": "thx", "thanks": "thx",
    "want to": "wanna", "going to": "gonna", "can you": "can u"
}

def augment_text_light(text):
    if not isinstance(text, str) or not text.strip():
        return text
    text = str(text)
    for word, abbr in ABBREVS.items():
        if random.random() < 0.25:
            text = re.sub(r"\b" + re.escape(word) + r"\b", abbr, text, flags=re.IGNORECASE)
    r = random.random()
    if r < 0.25:
        text = text.rstrip(".!?") + random.choice(["!!", "???", "?!"])
    elif r < 0.45:
        words = text.split()
        if words:
            idx = random.randint(0, len(words)-1)
            words[idx] = words[idx].upper()
            text = " ".join(words)
    elif r < 0.60:
        text = re.sub(r"[,.!?;:]", "", text)
    return re.sub(r"\s+", " ", text).strip()

hard_train = X_train_base[X_train_base["is_hard"] == True].copy()
hard_train["instruction_clean"] = hard_train["instruction_clean"].apply(augment_text_light)
hard_train["source"] = "augmented_hard"

easy_sample = X_train_base[X_train_base["is_hard"] == False].sample(
    frac=0.20,
    random_state=RANDOM_SEED
).copy()
easy_sample["instruction_clean"] = easy_sample["instruction_clean"].apply(augment_text_light)
easy_sample["source"] = "augmented_easy"

X_train = pd.concat([X_train_base, hard_train, easy_sample], ignore_index=True)
X_train = X_train.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print("Après augmentation:")
print(f"  Train final : {len(X_train):,}")
print(f"    Base      : {len(X_train_base):,}")
print(f"    Hard aug  : {len(hard_train):,}")
print(f"    Easy aug  : {len(easy_sample):,}")


Après augmentation:
  Train final : 83,156
    Base      : 63,852
    Hard aug  : 8,167
    Easy aug  : 11,137


## 9. Sauvegarde des splits

In [22]:
X_train.to_csv(DATA_SPLITS / "train.csv", index=False)
X_val.to_csv(DATA_SPLITS / "val.csv", index=False)
X_test.to_csv(DATA_SPLITS / "test.csv", index=False)

X_train.to_parquet(DATA_SPLITS / "train.parquet", index=False)
X_val.to_parquet(DATA_SPLITS / "val.parquet", index=False)
X_test.to_parquet(DATA_SPLITS / "test.parquet", index=False)

print("Splits sauvegardés:")
print(f"  train.csv: {len(X_train):,}")
print(f"  val.csv  : {len(X_val):,}")
print(f"  test.csv : {len(X_test):,}")
print()
print("Sources dans train:")
print(X_train["source"].value_counts().to_string())
print()
print("Sources dans val:")
print(X_val["source"].value_counts().to_string())
print()
print("Sources dans test:")
print(X_test["source"].value_counts().to_string())


Splits sauvegardés:
  train.csv: 83,156
  val.csv  : 6,655
  test.csv : 6,672

Sources dans train:
source
retail                    31152
support                   24533
augmented_easy            11137
augmented_hard             8167
amazon_single_qna          6918
amazon_multi_questions      909
synthetic_ecommerce         288
ecommerce_faq                52

Sources dans val:
source
retail    6655

Sources dans test:
source
retail    6672


## 10. Vérifications finales

In [24]:
print("Distribution labels train / val / test")
for name, split_df in [("train", X_train), ("val", X_val), ("test", X_test)]:
    counts = split_df["intent"].value_counts()
    print(f"{name:<5}: n={len(split_df):,}, intents={split_df['intent'].nunique()}, min={counts.min()}, max={counts.max()}")

print("Exemples train:")
display(X_train[["instruction_clean", "intent", "source", "is_hard"]].sample(10, random_state=RANDOM_SEED))


Distribution labels train / val / test
train: n=83,156, intents=66, min=644, max=14361
val  : n=6,655, intents=46, min=108, max=149
test : n=6,672, intents=46, min=108, max=150
Exemples train:


,instruction_clean,intent,source,is_hard
60278,I sent a product back but I hacen't received t...,track_order,retail,False
1389,where can I receive your company newsletter?,newsletter_subscription,support,False
5374,need to add items to the cart where do i do it,add_product,retail,False
67776,Anyone use this with a Lifeproof Fre case for ...,product_information,augmented_hard,True
66192,How much weight can it hold!!,product_information,augmented_hard,True
35510,I have a truble editing my shipping address,change_shipping_address,support,False
14649,i got to send an email to client service how c...,customer_service,retail,False
43264,i want my order history help me review it,order_history,retail,False
47308,help me to buy some products,place_order,support,False
17128,id like to check the availability of items in ...,availability_in_store,retail,False
